In [1]:
from modules.data_processing import *

root_path = os.getcwd()
train_df = load_pkl(os.path.join(root_path, 'data/svo/svo_train_dfp.pkl'))

/Users/tls/qml/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
from modules.quantum import *
obmap = {'n': 1, 'p': 1, 's': 1, 'out': 9}
nl = 1
ansatz = IQPAnsatz(obmap, nl)

In [3]:
from modules.functor import * 

einsum_arr = tree2tn(train_df, ['tree'])
qc_uncurried = tn2qc(einsum_arr, ansatz)
qc_curried = tn2qc(einsum_arr, ansatz, True)

100%|██████████| 5267/5267 [00:01<00:00, 3007.14it/s]


In [4]:
from collections import Counter

def test_valid_qc(qc_arr):
    i = 0
    for einsum_arr, param_arr in tqdm(qc_arr):
        count_dict = Counter(sum(einsum_arr[0],[]))
        out_count = 0
        for key, val in count_dict.items():
            if val == 1:
                out_count +=1
        if out_count != 9:
            print(f"FUCK {out_count} ({i})")
            break
        i += 1
    print("ALL GOOD!")
    
test_valid_qc(sum(qc_uncurried, []))
test_valid_qc(sum(qc_curried, []))

100%|██████████| 5267/5267 [00:00<00:00, 33723.67it/s]


ALL GOOD!


100%|██████████| 5267/5267 [00:00<00:00, 35052.80it/s]

ALL GOOD!


In [5]:
from modules.tensor_network import *
info1 = tn_metadata(sum(qc_uncurried,[]))
print("Qubit Count, Gate Count, Circuit Depth, Max Width")
print(f"Uncurried: {info1['avg']}")
info2 = tn_metadata(sum(qc_curried,[]))
print(f"Curried: {info2['avg']}")

100%|██████████| 5267/5267 [00:00<00:00, 27116.53it/s]


Qubit Count, Gate Count, Circuit Depth, Max Width
Uncurried: (21, 81, 17, 10)


100%|██████████| 5267/5267 [00:00<00:00, 38611.27it/s]

Curried: (15, 87, 24, 10)
